# Multivariate anomalies with TorchSharp
**EN** — An online autoencoder learns how speed, temperature and door state move together and flags readings that break the pattern.

**ID** — Autoencoder online mempelajari hubungan kecepatan, suhu, dan status pintu, lalu menandai pembacaan yang melanggar pola tersebut.

*Dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*

In [ ]:
// Using a local build instead of nuget.org? Uncomment and point to <repo>/artifacts/nuget (absolute path):
// #i "nuget: C:/src/bigpipe/artifacts/nuget"
#r "nuget: BigPipe.Analytics.Torch, 0.1.0"
#r "nuget: TorchSharp-cpu, 0.107.0"
using BigPipe.Client;
using BigPipe.Client.Admin;
using BigPipe.Analytics;
var bootstrap = Environment.GetEnvironmentVariable("BIGPIPE_BOOTSTRAP") ?? "localhost:9092";
var admin = new BigPipeAdminClient(Environment.GetEnvironmentVariable("BIGPIPE_ADMIN") ?? "http://localhost:9644");
var http = new BigPipeHttpClient(Environment.GetEnvironmentVariable("BIGPIPE_HTTP") ?? "http://localhost:8082");
string Unique(string p) => $"nb-{p}-{Guid.NewGuid().ToString("N")[..6]}";
var cluster = await admin.GetClusterAsync();
Console.WriteLine($"Connected to {cluster.ClusterId} (BigPipe {cluster.Version}) — {cluster.Credit}");

In [ ]:
using BigPipe.Analytics.Torch;
var topic = Unique("fleet");
await admin.CreateTopicAsync(topic, 1);
var rng = new Random(9);
var rows = new List<HttpProduceRecord>();
for (var i = 0; i < 1500; i++)
{
    var moving = rng.NextDouble() > 0.3;
    var door = moving ? 0 : rng.Next(2);
    if (i is 1300 or 1420) door = 1;   // door open while driving
    rows.Add(new HttpProduceRecord { Key = $"truck-{i % 5}", Value = new { speed = moving ? 40 + rng.NextDouble() * 40 : 0, temp_c = 4 + rng.NextDouble() + door * 2, door } });
}
foreach (var chunk in rows.Chunk(500)) await http.ProduceAsync(topic, chunk);
var detector = new TorchAutoencoderDetector(features: 3, k: 5, warmup: 300);
var found = new List<object>();
var n = 0;
await foreach (var r in BigPipeSource.Kafka(bootstrap, topic, from: OffsetReset.Earliest))
{
    var s = detector.Observe([(float)r.Num("speed"), (float)r.Num("temp_c"), (float)r.Num("door")]);
    if (s.IsAnomaly) found.Add(new { reading = n, r.Value, error = Math.Round(s.Value, 4) });
    if (++n == rows.Count) break;
}
found